# 04. Scoreboard (Text Cell)

ADR-016/017 §6 검증. 02의 DSC + 03의 train metric을 join → r 분석.

합격 기준:
- Pearson r(DSC, accuracy/R²) ≥ 0.4
- Spearman ρ ≥ 0.4
- Polluter hold-out 4/5 PASS
- 모델 5/5 양의 r

회귀 트랙은 ADR-012 Degradation Index 보조 보고.


In [ ]:
# ============================================================
# 0. Drive 마운트 + dsc/ 자동 검색 + sys.path 등록
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, sys, glob, json
import numpy as np
import pandas as pd


def _find_dsc_base():
    root = '/content/drive/MyDrive'
    if not os.path.isdir(root):
        return None
    for c in [f'{root}/capstone/dsc', f'{root}/dsc', f'{root}/capstone-dsc']:
        if os.path.isfile(f'{c}/dsc_framework/__init__.py'):
            return c
    for pat in [f'{root}/*/dsc_framework/__init__.py',
                f'{root}/*/*/dsc_framework/__init__.py',
                f'{root}/*/*/*/dsc_framework/__init__.py']:
        for hit in glob.glob(pat):
            return os.path.dirname(os.path.dirname(hit))
    return None


BASE = _find_dsc_base()
if BASE is None:
    drive_root = '/content/drive/MyDrive'
    listing = os.listdir(drive_root) if os.path.isdir(drive_root) else []
    raise RuntimeError(
        'dsc_framework/ 폴더를 G드라이브에서 못 찾음.\n'
        '  1) G드라이브 클라이언트 sync 완료 확인 (commit 직후면 잠시 대기 후 재시도)\n'
        '  2) Drive 마운트 확인 — !ls /content/drive/MyDrive\n'
        f'  현재 Drive 내용: {listing[:20]}'
    )

# 누락 파일 진단 — partial sync 시 빠른 실패
REQUIRED = ['shared_metrics.py', 'classification_cell.py', 'regression_cell.py',
            'image_cell.py', 'text_cell.py', 'text_cell_regression.py',
            'text_trainers.py', 'data_type_detection.py', 'router.py',
            'text_polluters', 'image_polluters']
missing = [f for f in REQUIRED if not os.path.exists(f'{BASE}/dsc_framework/{f}')]
if missing:
    raise RuntimeError(
        f'dsc_framework/ 파일 누락: {missing}\n'
        '→ G드라이브 sync 미완료. 잠시 대기 후 재실행.\n'
        '→ Colab Drive view stale 시: drive.flush_and_unmount() 후 재마운트.'
    )

RESULTS_DIR = f'{BASE}/results'
DATA_DIR = f'{BASE}/data/text'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

if BASE not in sys.path:
    sys.path.insert(0, BASE)

print(f'BASE: {BASE}')
print(f'dsc_framework 파일: {sorted(f for f in os.listdir(f"{BASE}/dsc_framework") if not f.startswith("_"))}')


In [ ]:
# ============================================================
# results csv 로드
# ============================================================
from scipy.stats import pearsonr, spearmanr

dsc_sweep = pd.read_csv(f'{RESULTS_DIR}/text_dsc_sweep.csv')
train_m = pd.read_csv(f'{RESULTS_DIR}/text_train_metrics.csv')

print('dsc_sweep:', dsc_sweep.shape, dsc_sweep['task'].value_counts().to_dict())
print('train_m  :', train_m.shape, train_m['task'].value_counts().to_dict())

JOIN_KEYS = ['dataset', 'polluter', 'level', 'seed']
# DSC는 multi-seed, train은 single-seed → train의 seed로 join
train_seed = train_m['seed'].iloc[0]
dsc_for_join = dsc_sweep[dsc_sweep['seed'] == train_seed]
print(f'join seed={train_seed}, dsc rows for join={len(dsc_for_join)}')


## 1. dataset별 r(DSC, metric) — 합격선 검토


In [ ]:
def r_per_dataset(dsc_df, train_df):
    j = dsc_df.merge(train_df, on=JOIN_KEYS)
    rows = []
    for ds_name in j['dataset'].unique():
        sub = j[j['dataset'] == ds_name]
        if len(sub) < 4 or sub['metric'].isna().all():
            rows.append({'dataset': ds_name, 'n': len(sub), 'note': 'insufficient'})
            continue
        sub_clean = sub.dropna(subset=['metric'])
        r_p, p_p = pearsonr(sub_clean['dsc_score'], sub_clean['metric'])
        r_s, p_s = spearmanr(sub_clean['dsc_score'], sub_clean['metric'])
        rows.append({
            'dataset': ds_name, 'n': len(sub_clean),
            'pearson': round(r_p, 4), 'p_pearson': p_p,
            'spearman': round(r_s, 4), 'p_spearman': p_s,
            'pass_r040': bool(r_p >= 0.4 and r_s >= 0.4),
        })
    return pd.DataFrame(rows)

print('=== 분류 트랙 ===')
print(r_per_dataset(dsc_for_join[dsc_for_join['task'] == 'classification'],
                   train_m[train_m['task'] == 'classification']))
print('\n=== 회귀 트랙 ===')
print(r_per_dataset(dsc_for_join[dsc_for_join['task'] == 'regression'],
                   train_m[train_m['task'] == 'regression']))


## 2. Polluter hold-out (4/5 PASS)


In [ ]:
def polluter_holdout(dsc_df, train_df):
    j = dsc_df.merge(train_df, on=JOIN_KEYS).dropna(subset=['metric'])
    polluters = j['polluter'].unique()
    rows = []
    for held in polluters:
        for ds_name in j['dataset'].unique():
            sub = j[(j['polluter'] != held) & (j['dataset'] == ds_name)]
            if len(sub) < 4:
                continue
            r, _ = pearsonr(sub['dsc_score'], sub['metric'])
            rows.append({'held_out': held, 'dataset': ds_name,
                         'pearson': round(r, 4), 'pass': r >= 0.4})
    return pd.DataFrame(rows)

ho_cls = polluter_holdout(dsc_for_join[dsc_for_join['task'] == 'classification'],
                          train_m[train_m['task'] == 'classification'])
ho_reg = polluter_holdout(dsc_for_join[dsc_for_join['task'] == 'regression'],
                          train_m[train_m['task'] == 'regression'])
print('=== 분류 hold-out ===')
print(ho_cls.groupby(['dataset']).agg(pass_count=('pass', 'sum'), total=('pass', 'count')))
print('\n=== 회귀 hold-out ===')
print(ho_reg.groupby(['dataset']).agg(pass_count=('pass', 'sum'), total=('pass', 'count')))


## 3. 모델별 r (5/5 양의 r)


In [ ]:
def r_per_model(dsc_df, train_df):
    j = dsc_df.merge(train_df, on=JOIN_KEYS).dropna(subset=['metric'])
    rows = []
    for m in j['model'].unique():
        sub = j[j['model'] == m]
        if len(sub) < 4:
            continue
        r, p = pearsonr(sub['dsc_score'], sub['metric'])
        rows.append({'model': m, 'pearson': round(r, 4), 'p': p, 'positive': r > 0})
    return pd.DataFrame(rows)

print('=== 분류 모델별 r ===')
print(r_per_model(dsc_for_join[dsc_for_join['task'] == 'classification'],
                  train_m[train_m['task'] == 'classification']))
print('\n=== 회귀 모델별 r ===')
print(r_per_model(dsc_for_join[dsc_for_join['task'] == 'regression'],
                  train_m[train_m['task'] == 'regression']))


## 4. Default vs tuned 가중치 grid search

이미지 cell `04` 패턴 미러. dead/live 메트릭 진단 + grid로 r 상승 가중치 탐색.

ADR-015 원칙: 본 grid 결과는 fallback 가중치 갭 분석. 운영·검증 정식 가중치는
Phase 4의 LLM weight generator 출력.


In [ ]:
# dead 메트릭 진단 — std < 0.01 (변동 없음 → r에 영향 없음)
METRIC_KEYS_CLS = ['completeness_text', 'uniqueness', 'validity', 'consistency',
                   'outlier_ratio', 'class_balance', 'feature_correlation',
                   'label_consistency', 'feature_informativeness', 'sample_quality_text']
METRIC_KEYS_REG = ['completeness_text', 'uniqueness', 'validity', 'consistency',
                   'outlier_ratio', 'target_distribution_quality',
                   'feature_correlation', 'target_smoothness',
                   'feature_informativeness_reg', 'sample_quality_text']

for task, keys in [('classification', METRIC_KEYS_CLS), ('regression', METRIC_KEYS_REG)]:
    sub = dsc_sweep[dsc_sweep['task'] == task]
    print(f'\n=== {task} ===')
    for k in keys:
        if k in sub.columns:
            print(f'  {k:30s} std={sub[k].std():.4f}  range=[{sub[k].min():.3f}, {sub[k].max():.3f}]')


In [ ]:
# 가중치 grid search — image cell의 score_image_v2.py 패턴 이식 예정.
# 본 셀은 stub: scipy.optimize.minimize로 -r 최소화 → tuned 가중치.
# from scipy.optimize import minimize
# (구현은 결과 분포 보고 결정)
print('grid search: results/text_dsc_sweep.csv 분석 후 구현')


---

**Phase 4 진입 전제**: 위 1~3 합격 시 plan 20260528-02 LLM prompt freeze + held-out 측정.
